In [ ]:
# ============================================================
# 📨 Email Summarizer & Context-Aware Prioritizer System
# ============================================================
# Project: Agentic AI Email Assistant (Final Version)
# Author: [Irimiea/McInnis]
# Course: [AI 471]
# Date: November 2025
#
# Description:
# This notebook demonstrates a modular, agentic AI system that processes
# incoming emails, summarizes their content, filters out spam, and assigns
# personalized priority levels based on relevance to the user. It uses
# a small Hugging Face model (BART) for summarization and integrates multiple
# specialized tools to simulate intelligent, context-aware reasoning.
#
# This version incorporates instructor feedback to go beyond a simple
# summarizer/ranker. The model now considers user profiles, spam likelihood,
# and message relevance — avoiding false "urgent" flags and reflecting
# personalized importance (e.g., events relevant to the user).
# ============================================================


# =========================
# 1. Import Dependencies
# =========================
# Required libraries for summarization, text processing, and data handling

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import pandas as pd


# =========================
# 2. Initialize Model (Fast, Reliable)
# =========================
# Using a lightweight summarization model for fast local execution.
# "facebook/bart-large-cnn" is optimized for text summarization tasks.

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)


# =========================
# 3. Define EmailParserTool
# =========================
# Parses a mock raw email into structured data fields.
# In a real-world setting, this would extract data from .eml or IMAP feeds.

def EmailParserTool(raw_email: str):
    """
    Extract structured information from raw email text.
    """
    parsed_email = {
        "sender": "project.lead@usafa.edu",
        "subject": "Updated Meeting Schedule",
        "timestamp": "2025-11-02 14:30",
        "body": raw_email.strip()
    }
    return parsed_email


# =========================
# 4. Define UserProfileTool
# =========================
# Creates a lightweight "user profile" defining the user's interests and roles.
# This allows the priority agent to factor in personalized relevance.

def UserProfileTool():
    """
    Defines key user interests, responsibilities, and sender importance.
    """
    user_profile = {
        "name": "Cadet McInnis",
        "interests": ["squadron", "inspection", "briefing", "training", "meeting"],
        "important_senders": ["commander", "lead", "instructor", "supervisor"]
    }
    return user_profile


# =========================
# 5. Define SpamFilterTool
# =========================
# Prevents false "High Priority" classification for spammy emails
# by checking for excessive urgency phrases and suspicious patterns.

def SpamFilterTool(email_text: str):
    """
    Returns True if the message is likely spam.
    Uses keyword frequency and phrasing heuristics.
    """
    text = email_text.lower()
    urgency_count = text.count("urgent")
    spam_keywords = ["win", "lottery", "credit", "click", "subscribe", "promotion"]
    if urgency_count > 2 or any(word in text for word in spam_keywords):
        return True
    return False


# =========================
# 6. Define SummaryGeneratorTool
# =========================
# Produces a concise summary for each email using the summarization model.

def SummaryGeneratorTool(email_body: str, max_length: int = 30):
    """
    Summarizes an email body using a small language model.
    """
    prompt = f"Summarize this email briefly: {email_body}"
    summary = summarizer(prompt, max_length=max_length, min_length=5, do_sample=False)[0]['summary_text']

    return summary


# =========================
# 7. Define PriorityClassifierTool
# =========================
# Combines keyword, sender, user profile, and spam analysis to determine
# personalized priority. This reflects the instructor’s feedback to handle
# contextual meaning and spam mitigation.

def PriorityClassifierTool(email, user_profile):
    """
    Assigns a personalized priority (High, Medium, Low) based on:
      - urgency words
      - sender importance
      - user interests
      - spam likelihood
    """
    text = (email["subject"] + " " + email["body"]).lower()

    # If spam detected, automatically set to Low
    if SpamFilterTool(text):
        return "Low"

    # Base priority by keywords
    high_keywords = ["urgent", "asap", "immediately", "deadline"]
    medium_keywords = ["reminder", "update", "schedule", "please", "today"]
    priority = "Low"

    if any(word in text for word in high_keywords):
        priority = "High"
    elif any(word in text for word in medium_keywords):
        priority = "Medium"

    # Adjust by sender importance
    if any(name in email["sender"] for name in user_profile["important_senders"]):
        if priority == "Low":
            priority = "Medium"
        elif priority == "Medium":
            priority = "High"

    # Adjust by user interest relevance
    if any(topic in text for topic in user_profile["interests"]):
        if priority == "Low":
            priority = "Medium"

    return priority


# =========================
# 8. Combine Tools into Workflow
# =========================
# This simulates the full system running on a set of mock emails.
# Each email goes through parsing, summarization, and classification.

emails = [
    "Hey, just a reminder that our project meeting moved from 1500 to 1530 today. Please bring your slides.",
    "URGENT: Squadron inspection report due ASAP. Submit your section immediately.",
    "Just FYI, team lunch moved to tomorrow.",
    "Congratulations! You won a free AirPods giveaway! URGENT, URGENT! Click now!",
    "Training update: your briefing slot has changed to 0800 hours tomorrow."
]

user_profile = UserProfileTool()

records = []
for e in emails:
    parsed_email = EmailParserTool(e)
    summary = SummaryGeneratorTool(parsed_email["body"])
    priority = PriorityClassifierTool(parsed_email, user_profile)
    records.append({
        "Sender": parsed_email["sender"],
        "Subject": parsed_email["subject"],
        "Summary": summary,
        "Priority": priority
    })


# =========================
# 9. Display Results
# =========================
# Presents the summarized results in a readable table.

df = pd.DataFrame(records)
print("=== Context-Aware Email Summary and Priority Results ===")
df


# =========================
# 10. Save Results
# =========================
# Saves the final processed table for recordkeeping.

import os

# Automatically find the user's Documents folder
documents_path = os.path.join(os.path.expanduser("~"), "Documents")

# Create the full file path
file_path = os.path.join(documents_path, "email_summary_priority_results.csv")

# Save the file there
df.to_csv(file_path, index=False)

print(f"✅ Results saved successfully to: {file_path}")
